# Module 00 — Foundations

Welcome! This is the very first lesson. By the end you'll understand the words everyone throws around (LLM, token, prompt, provider, framework) and you'll make your **first AI call**.

**How to use a notebook:** read each markdown cell, then run the code cell under it with **Shift + Enter**. Run them top to bottom, in order.

## 1. What is an LLM?

**LLM = Large Language Model.** Think of it as an extremely powerful autocomplete.

You give it some text (a **prompt**), and it predicts the most likely text that should come next, one small piece at a time. That's the whole trick. There is no database of facts inside it, and it does not "think" the way a human does — it has read an enormous amount of text and learned the patterns of language extraordinarily well.

Famous LLMs: **Gemini** (Google), **GPT** (OpenAI), **Llama** (Meta, often run via Groq).

### The one idea to remember: an LLM is **stateless**

It does **not** remember previous calls. Every time you call it, it only knows what you send in *that* request. Later, when we build "memory" and "agents," all we're really doing is cleverly deciding *what text to send back in* each time. Keep this in your pocket — it explains most of the course.

## 2. Tokens

Models don't read letters or words — they read **tokens**. A token is a chunk of text, roughly ¾ of a word.

- `"hello"` → 1 token
- `"internationalization"` → several tokens
- Rule of thumb: **1 token ≈ 4 characters ≈ 0.75 words** in English.

Why you should care:
- **Cost** — paid models charge per token (input tokens + output tokens).
- **Limits** — every model has a maximum number of tokens it can handle in one call (the **context window**).

You won't count tokens by hand — the library reports them for you, as you'll see at the end.

## 3. What is a framework (LangChain), and why not just call the API directly?

Every provider (Google, OpenAI, Groq) ships its own SDK, with its own function names and message formats. If you wrote your code against OpenAI and later wanted Gemini, you'd rewrite a lot of it.

A **framework** is a reusable layer that hides those differences. **LangChain** is that layer for LLMs: you learn *one* set of commands (`invoke`, `stream`, `bind_tools`, `create_agent`, …) and they work no matter which provider you choose.

Just as important: LangChain gives us ready-made building blocks for the advanced parts of this course — **tools, memory, RAG, and agents** — so we compose instead of reinventing. That's why the whole course is built on it.

> You *can* call a provider's API directly, and later you'll understand exactly what LangChain does for you under the hood. We start with the framework so you can build real things fast.

## 4. API keys and the `.env` file

To use a hosted model you need an **API key** — a secret password that proves you're allowed to call it (and is how usage gets billed).

We keep secrets in a file named `.env` in the **project root**, and **never** hard-code them in notebooks (so they don't accidentally end up on GitHub). The `python-dotenv` library loads that file into environment variables.

Your `.env` should contain at least:

```
GOOGLE_API_KEY=your_key_here
```

Get a free Gemini key at https://aistudio.google.com/app/apikey if you don't have one yet.

In [ ]:
import os
from dotenv import load_dotenv

# Load the secrets from the .env file in the project root (one folder up from this module).
load_dotenv(dotenv_path=os.path.join("..", ".env"))

# Use the API-key route for Gemini (not Vertex AI project auth).
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

# Confirm the key loaded WITHOUT printing the secret itself.
key = os.getenv("GOOGLE_API_KEY")
print("GOOGLE_API_KEY found:", bool(key))
if key:
    print("It starts with:", key[:4] + "...")  # safe preview only

: 

## 5. Your first model call

Now the moment of truth. We will:
1. Create a model with `init_chat_model` (LangChain's universal way to load *any* provider's model).
2. Send it a prompt with `.invoke(...)`.
3. Print the reply.

The string `"google_genai:gemini-2.5-flash"` means **provider `google_genai`, model `gemini-2.5-flash`**. To use OpenAI later you'd write `"openai:gpt-4o-mini"` — *same code, different string.* That's the framework paying off already.

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash")

response = model.invoke("In one short sentence, explain what an LLM is to a 10-year-old.")

print(response.text)

## 6. What did we actually get back?

`.invoke()` returns an **`AIMessage`** object, not just a plain string. The text is the part you read, but it also carries useful metadata — including the **token usage** we talked about in section 2.

Run this to peek inside:

In [ ]:
print("Type of response:", type(response).__name__)
print()
print("The text:", response.text)
print()
print("Token usage:", response.usage_metadata)

## Recap

- An **LLM** is a powerful text-predictor; it is **stateless** (no memory between calls).
- Models read **tokens**; tokens drive cost and limits.
- A **framework** (LangChain) gives us one consistent API across all providers, plus building blocks for tools/memory/RAG/agents.
- Secrets live in **`.env`**, loaded with `python-dotenv` — never hard-coded.
- `init_chat_model(...)` + `.invoke(...)` = your first AI call.
- `.invoke()` returns an **`AIMessage`** with `.text` and `.usage_metadata`.

## 🛠️ Try it yourself

1. Change the prompt in section 5 to ask the model something *you* are curious about. Re-run it.
2. Ask it the **same** question twice in two separate `.invoke()` calls. Do you get identical answers? Why might they differ even though it's the same model? (Hint: think about what "predict the *most likely* next text" really means.)
3. Look at `usage_metadata` and find how many **output tokens** your answer used. Make the model answer in a single word and watch that number drop.
4. **Stretch:** try swapping the model string to another provider you have a key for (e.g. `"groq:llama-3.3-70b-versatile"`). Notice that *only the string changed* — that's the framework doing its job.

When you're done, say **"next"** and we'll build **Module 01 — Models**.